In [7]:
import cv2
import mediapipe as mp
import argparse
from collections import deque


In [ ]:
class JumpCounter:
    """Estimate completed jumps from smoothed hip vertical movement."""

    def __init__(self, threshold=0.025, smoothing=5, cooldown=5):
        self.count = 0
        self.threshold = threshold
        self.history = deque(maxlen=smoothing)
        self.cooldown = cooldown
        self.cooldown_left = 0
        self.baseline = None
        self.airborne = False
        self.minimum_rise = 0.0

    def update(self, hip_y):
        self.history.append(hip_y)
        smooth_y = sum(self.history) / len(self.history)

        if self.baseline is None:
            self.baseline = smooth_y
            return self.count, "Calibrating"

        if self.cooldown_left > 0:
            self.cooldown_left -= 1

        rise = self.baseline - smooth_y  # smaller y means higher in image

        if not self.airborne:
            self.baseline = 0.98 * self.baseline + 0.02 * smooth_y
            if rise >= self.threshold and self.cooldown_left == 0:
                self.airborne = True
                self.minimum_rise = rise
                return self.count, "Jump detected"
            return self.count, "Ready"

        self.minimum_rise = max(self.minimum_rise, rise)
        if rise < self.threshold * 0.35:
            if self.minimum_rise >= self.threshold:
                self.count += 1
                self.cooldown_left = self.cooldown
            self.airborne = False
            self.minimum_rise = 0.0
            return self.count, "Landing"

        return self.count, "In air"


def resize_to_fit(frame, max_width=1100, max_height=750):
    """Fit the complete frame inside the display area without cropping.

    Upscaling is allowed for small videos so the person is easier to see.
    The original aspect ratio is always preserved.
    """
    height, width = frame.shape[:2]
    if width <= 0 or height <= 0:
        return frame

    scale = min(max_width / width, max_height / height)
    new_width = max(1, int(width * scale))
    new_height = max(1, int(height * scale))
    interpolation = cv2.INTER_AREA if scale < 1 else cv2.INTER_LINEAR
    return cv2.resize(frame, (new_width, new_height), interpolation=interpolation)


def run(source, display_width=1100, display_height=750):
    mp_pose = mp.solutions.pose
    mp_draw = mp.solutions.drawing_utils
    cap = cv2.VideoCapture(source)

    if not cap.isOpened():
        print("Error: Could not open video source.")
        return

    window_name = "AI Skipping Rope Counter (press q to quit)"
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(window_name, display_width, display_height)

    counter = JumpCounter(threshold=0.025, smoothing=5, cooldown=5)

    with mp_pose.Pose(
        static_image_mode=False,
        model_complexity=1,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    ) as pose:
        while cap.isOpened():
            success, frame = cap.read()
            if not success:
                break

            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = pose.process(rgb)
            status = "Pose not detected"

            if results.pose_landmarks:
                landmarks = results.pose_landmarks.landmark
                left_hip = landmarks[mp_pose.PoseLandmark.LEFT_HIP.value]
                right_hip = landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value]

                if min(left_hip.visibility, right_hip.visibility) > 0.5:
                    hip_y = (left_hip.y + right_hip.y) / 2
                    count, status = counter.update(hip_y)
                else:
                    count = counter.count

                mp_draw.draw_landmarks(
                    frame, results.pose_landmarks, mp_pose.POSE_CONNECTIONS
                )
            else:
                count = counter.count

            cv2.putText(
                frame, f"Jumps: {count}", (100, 100),
                cv2.FONT_HERSHEY_COMPLEX_SMALL, 5, (255, 255, 255), 5
            )
            cv2.putText(
                frame, f"Status: {status}", (100, 200),
                cv2.FONT_HERSHEY_COMPLEX_SMALL, 3, (0, 255, 255), 5
            )
            display_frame = resize_to_fit(frame, display_width, display_height)
            cv2.imshow(window_name, display_frame)

            if cv2.waitKey(1) & 0xFF == ord("q"):
                break

    cap.release()
    cv2.destroyAllWindows()
run('kid_jumping.mp4')
# run('jumping_03.mp4')
# run('jumping_04.mp4')

if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Pose-based jump counter")
    parser.add_argument(
        "--video", type=str, default=None,
        help="Path to a recorded video. Omit to use the webcam."
    )
    parser.add_argument(
        "--width", type=int, default=1100,
        help="Display window width in pixels (default: 1100)."
    )
    parser.add_argument(
        "--height", type=int, default=750,
        help="Display window height in pixels (default: 750)."
    )
    args = parser.parse_args()
    run(args.video if args.video else 0, args.width, args.height)


usage: ipykernel_launcher.py [-h] [--video VIDEO] [--width WIDTH]
                             [--height HEIGHT]
ipykernel_launcher.py: error: unrecognized arguments: --f=c:\Users\LENOVO\AppData\Roaming\jupyter\runtime\kernel-v37ff4473a9c209d13f323b9b4eb4741b015e78e14.json


SystemExit: 2